# LeetCode 3083 — Existence of a Substring in a String and Its Reverse bằng Python

Link đề: https://leetcode.com/problems/existence-of-a-substring-in-a-string-and-its-reverse/ · Độ khó: **Easy** · Chủ đề: Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def isSubstringPresent(self, s: str) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho chuỗi `s`, cần biết **có tồn tại một chuỗi con độ dài `2`** của `s` mà **cũng là chuỗi con của `reverse(s)`** (chuỗi `s` viết ngược) hay không.
2. Nắm công cụ chuỗi của Python: **đảo chuỗi** `s[::-1]`, **cắt lát** `s[i:i + 2]`, toán tử **`in`** cho chuỗi con, và `set` để lưu các cặp ký tự.
3. Đi từ cách **đảo chuỗi rồi kiểm tra từng cặp bằng `in`** → **so ký tự bằng tay** → **hai tập các cặp rồi giao nhau** → **một tập duy nhất: cặp `ab` cần có `ba`** → **bảng `26 x 26`**.
4. Hiểu mấu chốt: `"ab"` là chuỗi con của `reverse(s)` **khi và chỉ khi** `"ba"` là chuỗi con của `s`. Vậy ta chỉ cần xét **các cặp ký tự liền nhau của `s` và đảo ngược của chúng**.
5. Tránh các lỗi kinh điển: vượt biên khi cắt cặp cuối, nhầm với "`s` là palindrome", bỏ sót trường hợp hai ký tự liền nhau giống nhau (`"ee"`), và quên `len(s) == 1`.

## 2. Đề bài & ví dụ

Cho một chuỗi `s`. Hãy tìm xem có tồn tại **một chuỗi con độ dài `2`** của `s` mà **cũng xuất hiện trong `reverse(s)`** (chuỗi `s` viết theo thứ tự ngược lại) hay không.

Trả về `true` nếu tồn tại, ngược lại `false`.

(Chuỗi con là dãy ký tự **liên tiếp** trong chuỗi.)

**Ví dụ 1**

- Input: `s = "leetcode"`
- Output: `true`
- Giải thích: `reverse(s) = "edocteel"`. Chuỗi con `"ee"` (độ dài 2) xuất hiện trong cả `s` và `reverse(s)`.

**Ví dụ 2**

- Input: `s = "abcba"`
- Output: `true`
- Giải thích: `reverse(s) = "abcba"` (chính nó). Mọi chuỗi con độ dài 2 của `s` (như `"ab"`, `"bc"`, `"cb"`, `"ba"`) đều có trong `reverse(s)`.

**Ví dụ 3**

- Input: `s = "abcd"`
- Output: `false`
- Giải thích: `reverse(s) = "dcba"`. Các chuỗi con độ dài 2 của `s` là `"ab"`, `"bc"`, `"cd"`; của `reverse(s)` là `"dc"`, `"cb"`, `"ba"`. Không có cặp chung nào.

**Ràng buộc:** `1 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
import random
import string

examples = [
    ("leetcode", True),
    ("abcba", True),
    ("abcd", False),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected)

'leetcode' -> kỳ vọng True
'abcba' -> kỳ vọng True
'abcd' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Đảo chuỗi: `s[::-1]`

Cắt lát có dạng `s[bắt_đầu:kết_thúc:bước]`. Với **bước `-1`**, Python đi từ cuối về đầu, nên `s[::-1]` là chuỗi **viết ngược**. Chuỗi gốc không đổi (chuỗi là bất biến).

In [2]:
s = "leetcode"
r = s[::-1]
print(r)                 # edocteel
print(s)                 # leetcode (không đổi)
print("abcba"[::-1])     # abcba

edocteel
leetcode
abcba


### 3.2. Chuỗi con độ dài 2 (cặp ký tự liền nhau): `s[i:i + 2]`

Với chuỗi độ dài `n`, các chuỗi con độ dài `2` bắt đầu ở `i = 0, 1, ..., n - 2`, tức **`n - 1` cặp**. Nếu `i = n - 1` thì `s[i:i + 2]` chỉ có **một** ký tự (cắt lát không báo lỗi nhưng ngắn hơn mong đợi), nên vòng lặp phải dừng ở `range(n - 1)`.

In [3]:
s = "abcd"
print([s[i:i + 2] for i in range(len(s) - 1)])      # ['ab', 'bc', 'cd']
print(repr(s[3:5]))                                 # 'd' : chỉ 1 ký tự (cẩn thận!)
print([s[i:i + 2] for i in range(len("a") - 1)])    # [] : chuỗi 1 ký tự không có cặp nào

['ab', 'bc', 'cd']
'd'
[]


### 3.3. Toán tử `in` cho chuỗi con

`sub in text` là `True` nếu `sub` là **chuỗi con liên tiếp** của `text`. Khác với phép kiểm tra thuộc về của list. Cần phân biệt: `"ac" in "abc"` là `False` vì `a` và `c` không liền nhau.

In [4]:
print("ee" in "leetcode")      # True
print("ac" in "abc")           # False (không liền nhau)
print("dc" in "edocteel")      # False
print("ba" in "abcba")         # True

True
False
False
True


### 3.4. `set` để lưu các cặp và kiểm tra nhanh

`set` là tập hợp không trùng. Ta có thể gom **mọi cặp ký tự liền nhau** vào một `set` rồi tra `x in tập` với thời gian trung bình `O(1)`. Phép **giao** hai tập: `a & b` (các phần tử chung); `a & b` khác rỗng ⇒ có phần tử chung.

In [5]:
s = "abcd"
pairs = {s[i:i + 2] for i in range(len(s) - 1)}
print(pairs)                                   # {'ab', 'bc', 'cd'} (thứ tự có thể khác)
print("bc" in pairs)                           # True
a, b = {"ab", "bc"}, {"cb", "bc"}
print(a & b, bool(a & b))                      # {'bc'} True

{'bc', 'ab', 'cd'}
True
{'bc'} True


### 3.5. `any(...)` và mã ký tự `ord`

- `any(điều kiện for ...)` là `True` nếu **có ít nhất một** phần tử thỏa.
- `ord('a') = 97`, nên `ord(c) - ord('a')` đổi chữ thường thành số `0..25` — dùng được làm chỉ số cho bảng `26 x 26`.

In [6]:
print(any(x > 2 for x in [1, 2, 3]))      # True
print(any(x > 5 for x in [1, 2, 3]))      # False
print(ord("a"), ord("c") - ord("a"))      # 97 2

True
False
97 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "leetcode"`.

- Đảo: `r = "edocteel"`.
- Các cặp của `s`: `le`, `ee`, `et`, `tc`, `co`, `od`, `de`.
- Các cặp của `r`: `ed`, `do`, `oc`, `ct`, `te`, `ee`, `el`.
- Cặp chung: `ee` ⇒ `true`.

Nhận xét: cặp `ed` của `r` chính là cặp `de` của `s` **viết ngược**; cặp `do` của `r` là `od` của `s` ngược; ... Cứ cặp `xy` trong `r` tương ứng cặp `yx` trong `s`. Nghĩa là **tập cặp của `r` = tập các cặp của `s` đảo ngược từng cặp**.

In [7]:
s = "leetcode"
r = s[::-1]
pairs_s = [s[i:i + 2] for i in range(len(s) - 1)]
pairs_r = [r[i:i + 2] for i in range(len(r) - 1)]
print("cặp của s:", pairs_s)
print("cặp của r:", pairs_r)
print("cặp của s đảo từng cặp:", [p[::-1] for p in pairs_s])
print("tập cặp r == tập (cặp s đảo):", set(pairs_r) == {p[::-1] for p in pairs_s})
print("cặp chung:", set(pairs_s) & set(pairs_r))

cặp của s: ['le', 'ee', 'et', 'tc', 'co', 'od', 'de']
cặp của r: ['ed', 'do', 'oc', 'ct', 'te', 'ee', 'el']
cặp của s đảo từng cặp: ['el', 'ee', 'te', 'ct', 'oc', 'do', 'ed']
tập cặp r == tập (cặp s đảo): True
cặp chung: {'ee'}


## 5. Cách 1 — Brute Force / baseline: đảo chuỗi rồi hỏi `in`

**Ý tưởng.** Đúng như đề: tạo `r = s[::-1]`; với mỗi cặp liền nhau `s[i:i + 2]` (`i` từ `0` tới `n - 2`), nếu `cặp in r` thì trả `True`. Hết vòng mà chưa có thì `False`.

In [8]:
def is_substring_present_in(s: str) -> bool:
    r = s[::-1]                                   # chuỗi viết ngược
    for i in range(len(s) - 1):                   # mỗi cặp liền nhau của s
        pair = s[i:i + 2]
        if pair in r:                             # pair có là chuỗi con của r không?
            return True
    return False


for s, expected in examples:
    print(repr(s), is_substring_present_in(s), "(kỳ vọng", expected, ")")

'leetcode' True (kỳ vọng True )
'abcba' True (kỳ vọng True )
'abcd' False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Có `n - 1` cặp; mỗi lần `pair in r` tìm kiếm trong chuỗi dài `n` ⇒ `O(n)`. Tổng **Time `O(n²)`**; **Space `O(n)`** cho `r`. Với `n <= 100` là cực nhanh.

**Nút thắt:** kiểm tra `in` lặp lại nhiều lần trên cùng chuỗi `r`; ta có thể lưu sẵn các cặp của `r` vào `set` để tra `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát chính.** Cặp `xy` là chuỗi con của `reverse(s)` ⇔ có `i` sao cho `reverse(s)[i] = x` và `reverse(s)[i + 1] = y`. Vì `reverse(s)[i] = s[n - 1 - i]` nên điều này nghĩa là `s[n - 1 - i] = x` và `s[n - 2 - i] = y`, tức trong `s` hai ký tự **liền nhau** `y` rồi `x`: cặp **`yx`** là chuỗi con của `s`.

**Kết luận.** `xy ∈ pairs(reverse(s))` ⇔ `yx ∈ pairs(s)`.

Do đó bài toán ⇔ **tồn tại cặp `xy` của `s` mà `yx` cũng là cặp của `s`**. Khi `x == y` (hai ký tự liền nhau giống nhau, như `"ee"`), `yx = xy` nên điều kiện đúng ngay.

Nhờ đó ta **không cần dựng `reverse(s)`**: chỉ cần một `set` các cặp của `s`.

In [9]:
# Kiểm chứng khẳng định: xy in pairs(reverse(s)) <=> yx in pairs(s)
rng = random.Random(1)
for _ in range(1000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(2, 12)))
    pairs_s = {s[i:i + 2] for i in range(len(s) - 1)}
    r = s[::-1]
    pairs_r = {r[i:i + 2] for i in range(len(r) - 1)}
    assert pairs_r == {p[::-1] for p in pairs_s}
print("tập cặp của reverse(s) đúng bằng tập các cặp của s đảo ngược")

tập cặp của reverse(s) đúng bằng tập các cặp của s đảo ngược


## 7. Các cách tiếp theo

### Cách 2 — So sánh từng ký tự bằng tay (không dùng `in`)

**Giải thích.** `reverse(s)` chỉ là `s` đọc từ phải sang trái. Cặp `s[i]s[i + 1]` xuất hiện trong `reverse(s)` ở vị trí `j` nếu `s[i] == r[j]` và `s[i + 1] == r[j + 1]`. Ta viết hai vòng lặp lồng nhau, so từng ký tự. Cách này giúp thấy `in` thực chất làm gì.

In [10]:
def is_substring_present_manual(s: str) -> bool:
    r = s[::-1]
    n = len(s)
    for i in range(n - 1):                 # cặp s[i], s[i+1]
        for j in range(n - 1):             # vị trí j trong r
            if s[i] == r[j] and s[i + 1] == r[j + 1]:
                return True
    return False


for s, expected in examples:
    print(repr(s), is_substring_present_manual(s), "(kỳ vọng", expected, ")")

'leetcode' True (kỳ vọng True )
'abcba' True (kỳ vọng True )
'abcd' False (kỳ vọng False )


**Độ phức tạp Cách 2:** hai vòng lồng `O(n²)` với phép so sánh `O(1)` ⇒ **Time `O(n²)`**, **Space `O(n)`** (chuỗi `r`).

### Cách 3 — Hai tập cặp rồi giao

**Giải thích.** Gom cặp của `s` vào `pairs_s` và cặp của `reverse(s)` vào `pairs_r`. Nếu **giao khác rỗng** thì `True`. Mỗi cặp chỉ dựng một lần.

In [11]:
def is_substring_present_sets(s: str) -> bool:
    r = s[::-1]
    pairs_s = {s[i:i + 2] for i in range(len(s) - 1)}
    pairs_r = {r[i:i + 2] for i in range(len(r) - 1)}
    return len(pairs_s & pairs_r) > 0


for s, expected in examples:
    print(repr(s), is_substring_present_sets(s), "(kỳ vọng", expected, ")")

'leetcode' True (kỳ vọng True )
'abcba' True (kỳ vọng True )
'abcd' False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(n)`** (dựng hai tập, giao tập kích thước `O(n)`; trung bình), **Space `O(n)`**.

### Cách 4 — Một tập duy nhất: cặp `xy` cần `yx`

**Giải thích.** Dùng khẳng định ở §6: chỉ cần tập `pairs` của `s`, rồi kiểm tra có cặp `p` nào có `p[::-1]` cũng nằm trong `pairs`. Không cần chuỗi `reverse(s)`.

In [12]:
def is_substring_present_one_set(s: str) -> bool:
    pairs = {s[i:i + 2] for i in range(len(s) - 1)}
    return any(p[::-1] in pairs for p in pairs)


for s, expected in examples:
    print(repr(s), is_substring_present_one_set(s), "(kỳ vọng", expected, ")")

'leetcode' True (kỳ vọng True )
'abcba' True (kỳ vọng True )
'abcd' False (kỳ vọng False )


In [13]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành Solution khi nộp)
class SolutionOneSet:
    def isSubstringPresent(self, s: str) -> bool:
        pairs = {s[i:i + 2] for i in range(len(s) - 1)}
        return any(p[::-1] in pairs for p in pairs)


print(SolutionOneSet().isSubstringPresent("abcd"))      # False

False


**Độ phức tạp Cách 4:** **Time `O(n)`** (trung bình), **Space `O(n)`**. Số cặp phân biệt tối đa `min(n - 1, 26²)`.

### Cách 5 — Bảng đánh dấu `26 x 26`

**Giải thích.** Vì `s` chỉ gồm chữ thường, ta dùng bảng `seen[a][b]` (với `a, b` là số `0..25`) đánh dấu "đã thấy cặp `ab`". Duyệt từng cặp liền nhau `(a, b)`: đặt `seen[a][b] = True`, rồi nếu `seen[b][a]` cũng `True` thì trả `True` (kể cả khi `a == b`, vì vừa đặt xong). Không cần băm chuỗi, **Space `O(26²)`** hằng số.

In [14]:
def is_substring_present_table(s: str) -> bool:
    seen = [[False] * 26 for _ in range(26)]
    for i in range(len(s) - 1):
        a = ord(s[i]) - ord("a")
        b = ord(s[i + 1]) - ord("a")
        seen[a][b] = True                   # đã thấy cặp (a, b)
        if seen[b][a]:                      # đã thấy cặp (b, a) chưa? (a == b: vừa đặt)
            return True
    return False


for s, expected in examples:
    print(repr(s), is_substring_present_table(s), "(kỳ vọng", expected, ")")

'leetcode' True (kỳ vọng True )
'abcba' True (kỳ vọng True )
'abcd' False (kỳ vọng False )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (bảng cố định `26 x 26`). Đây là cách tối ưu về bộ nhớ, nhưng dựa vào việc bảng chữ cái nhỏ.

Vì sao dừng sớm vẫn đúng? Khi ta xét cặp `(a, b)` ở vị trí `i`, mọi cặp ở vị trí trước đã được đánh dấu. Nếu `(b, a)` đã thấy từ trước thì có cặp phản chiều; còn nếu cặp `(b, a)` xuất hiện **sau** thì khi tới đó, lúc đó `(a, b)` đã được đánh dấu và ta sẽ bắt được. Vậy mọi cặp phản chiều đều được phát hiện.

## 8. TRACE từng bước

Ta trace Cách 5 (bảng) và Cách 4 (một tập) trên `"abcba"` và trên `"abcd"` để thấy khi nào dừng.

In [15]:
def trace_table(s: str) -> bool:
    seen = set()                      # dùng set (a, b) để in cho dễ nhìn; tương đương bảng 26x26
    print("s =", repr(s))
    for i in range(len(s) - 1):
        a, b = s[i], s[i + 1]
        seen.add((a, b))
        found = (b, a) in seen
        print(f"i={i}: cặp ({a},{b}) -> đánh dấu; có ({b},{a}) chưa? {found} | seen={sorted(seen)}")
        if found:
            print("-> True")
            return True
    print("-> False")
    return False


assert trace_table("abcba") is True
print()
assert trace_table("abcd") is False

s = 'abcba'
i=0: cặp (a,b) -> đánh dấu; có (b,a) chưa? False | seen=[('a', 'b')]
i=1: cặp (b,c) -> đánh dấu; có (c,b) chưa? False | seen=[('a', 'b'), ('b', 'c')]
i=2: cặp (c,b) -> đánh dấu; có (b,c) chưa? True | seen=[('a', 'b'), ('b', 'c'), ('c', 'b')]
-> True

s = 'abcd'
i=0: cặp (a,b) -> đánh dấu; có (b,a) chưa? False | seen=[('a', 'b')]
i=1: cặp (b,c) -> đánh dấu; có (c,b) chưa? False | seen=[('a', 'b'), ('b', 'c')]
i=2: cặp (c,d) -> đánh dấu; có (d,c) chưa? False | seen=[('a', 'b'), ('b', 'c'), ('c', 'd')]
-> False


In [16]:
def trace_one_set(s: str) -> bool:
    pairs = {s[i:i + 2] for i in range(len(s) - 1)}
    print("pairs của", repr(s), "=", sorted(pairs))
    for p in sorted(pairs):
        print(f"  cặp {p!r}: đảo = {p[::-1]!r}, có trong pairs? {p[::-1] in pairs}")
        if p[::-1] in pairs:
            return True
    return False


assert trace_one_set("leetcode") is True
assert trace_one_set("abcd") is False

pairs của 'leetcode' = ['co', 'de', 'ee', 'et', 'le', 'od', 'tc']
  cặp 'co': đảo = 'oc', có trong pairs? False
  cặp 'de': đảo = 'ed', có trong pairs? False
  cặp 'ee': đảo = 'ee', có trong pairs? True
pairs của 'abcd' = ['ab', 'bc', 'cd']
  cặp 'ab': đảo = 'ba', có trong pairs? False
  cặp 'bc': đảo = 'cb', có trong pairs? False
  cặp 'cd': đảo = 'dc', có trong pairs? False


Bảng trace Cách 5 với `"abcba"`:

| i | cặp (a, b) | đánh dấu | (b, a) đã thấy? |
|---|---|---|---|
| 0 | (a, b) | {ab} | (b, a) chưa |
| 1 | (b, c) | {ab, bc} | (c, b) chưa |
| 2 | (c, b) | {ab, bc, cb} | (b, c) **có** ⇒ `True` |

Với `"abcd"`: các cặp `ab`, `bc`, `cd` không có phản chiều nào ⇒ hết vòng, `False`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (một tập các cặp): ngắn, `O(n)` và không cần dựng chuỗi đảo.

In [17]:
class Solution:
    def isSubstringPresent(self, s: str) -> bool:
        pairs = {s[i:i + 2] for i in range(len(s) - 1)}     # mọi cặp ký tự liền nhau
        return any(p[::-1] in pairs for p in pairs)         # cặp xy có cặp yx (kể cả xx)


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.isSubstringPresent(s))

'leetcode' -> True
'abcba' -> True
'abcd' -> False


## 10. Vì sao thuật toán đúng

- **Tương đương cốt lõi:** `xy` là chuỗi con của `reverse(s)` ⇔ `yx` là chuỗi con của `s` (đọc chuỗi ngược lại thì thứ tự hai ký tự liền nhau bị đảo). Do đó "tồn tại chuỗi con độ dài 2 chung của `s` và `reverse(s)`" ⇔ "tồn tại cặp `xy` của `s` mà `yx` cũng là cặp của `s`".
- **Trường hợp `x == y`:** `yx = xy`, và chính `xy` là một cặp của `s`, nên điều kiện đúng ngay. Đây là lý do `"leetcode"` (có `"ee"`) trả `true` mà không cần ký tự nào khác.
- **Chuỗi một ký tự:** không có cặp nào ⇒ tập rỗng ⇒ `any` trên tập rỗng là `False` ⇒ đúng (không có chuỗi con độ dài 2).
- **Palindrome:** nếu `s == reverse(s)` và `len(s) >= 2`, mọi cặp của `s` đều thuộc `reverse(s)` nên đáp án `true`. Chiều ngược lại **không** đúng: `"leetcode"` không là palindrome nhưng vẫn `true`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. `pair in reverse` | `O(n²)` | `O(n)` |
| 2. So ký tự bằng tay | `O(n²)` | `O(n)` |
| 3. Hai tập rồi giao | `O(n)` (trung bình) | `O(n)` |
| 4. Một tập: `xy` cần `yx` | `O(n)` (trung bình) | `O(n)` |
| 5. Bảng `26 x 26` | `O(n)` | `O(1)` |

Với `n <= 100` tất cả đều chạy tức thì.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "in": is_substring_present_in,
    "manual": is_substring_present_manual,
    "sets": is_substring_present_sets,
    "one_set": is_substring_present_one_set,
    "table": is_substring_present_table,
    "leetcode": sol.isSubstringPresent,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all("a", False, "một ký tự: không có chuỗi con độ dài 2")
check_all("aa", True, "hai ký tự giống nhau: aa là chung")
check_all("ab", False, "ab, reverse = ba: không chung")
check_all("aba", True, "palindrome độ dài 3")
check_all("abba", True, "palindrome độ dài chẵn (cũng có bb)")
check_all("abab", True, "ab và ba cùng xuất hiện trong s")
check_all("abc", False, "ba cặp không phản chiều")
check_all("abca", False, "ab, bc, ca: không có phản chiều")
check_all("abcdefghijklmnopqrstuvwxyz", False, "bảng chữ cái xuôi: không phản chiều")
check_all("zyxwvutsrqponmlkjihgfedcba", False, "bảng chữ cái ngược: cũng không")
check_all("abcdefghijklmnopqrstuvwxyzzyx", True, "có zz")
check_all("a" * 100, True, "100 chữ a")
check_all("ab" * 50, True, "abab...: có cả ab và ba")

PASS một ký tự: không có chuỗi con độ dài 2
PASS hai ký tự giống nhau: aa là chung
PASS ab, reverse = ba: không chung
PASS palindrome độ dài 3
PASS palindrome độ dài chẵn (cũng có bb)
PASS ab và ba cùng xuất hiện trong s
PASS ba cặp không phản chiều
PASS ab, bc, ca: không có phản chiều
PASS bảng chữ cái xuôi: không phản chiều
PASS bảng chữ cái ngược: cũng không
PASS có zz
PASS 100 chữ a
PASS abab...: có cả ab và ba


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **chỉ kiểm tra `s` là palindrome**; (b) **chỉ kiểm tra hai ký tự liền nhau giống nhau**.

In [20]:
def wrong_palindrome_only(s):
    return len(s) >= 2 and s == s[::-1]


def wrong_adjacent_equal_only(s):
    return any(s[i] == s[i + 1] for i in range(len(s) - 1))


bad1 = "leetcode"     # không phải palindrome, nhưng đáp án true
bad2 = "abab"         # không có ký tự giống nhau liền kề, nhưng đáp án true
print(wrong_palindrome_only(bad1), "!=", is_substring_present_in(bad1))
print(wrong_adjacent_equal_only(bad2), "!=", is_substring_present_in(bad2))
assert wrong_palindrome_only(bad1) != is_substring_present_in(bad1)
assert wrong_adjacent_equal_only(bad2) != is_substring_present_in(bad2)

False != True
False != True


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi đúng ràng buộc (độ dài `1..100`, chữ thường). Dùng bảng chữ cái nhỏ để dễ gặp trường hợp `true`, và bảng chữ cái đầy đủ cho trường hợp thưa. Tham chiếu: định nghĩa gốc bằng tập cặp của `s` và của `reverse(s)`.

In [21]:
def reference(s):
    r = s[::-1]
    return bool({s[i:i + 2] for i in range(len(s) - 1)} & {r[i:i + 2] for i in range(len(r) - 1)})


rng = random.Random(3083)
counts = {True: 0, False: 0}
for trial in range(3000):
    alphabet = rng.choice(["a", "ab", "abc", "abcd", string.ascii_lowercase])
    n = rng.choice([1, 2, 3, 5, 10, 26, 100])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    assert 1 <= len(s) <= 100 and s.islower()
    ref = reference(s)
    counts[ref] += 1
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
print("PASS 3000 chuỗi ngẫu nhiên; phân bố:", counts)
assert counts[True] > 100 and counts[False] > 100

PASS 3000 chuỗi ngẫu nhiên; phân bố: {True: 2053, False: 947}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cắt cặp ở vị trí cuối (vượt biên)

In [22]:
def sai_vuot_bien(s):
    r = s[::-1]
    for i in range(len(s)):               # SAI: i = len(s) - 1 cho cặp chỉ có 1 ký tự
        if s[i:i + 2] in r:
            return True
    return False


print("sai:", sai_vuot_bien("ab"), "| đúng:", Solution().isSubstringPresent("ab"))

sai: True | đúng: False


**Vì sao sai:** ở `i = n - 1`, `s[i:i + 2]` chỉ có **một** ký tự (`"b"`), và `"b" in "ba"` là `True`, làm kết quả sai. **Cách sửa:** `range(len(s) - 1)` để mọi lát cắt đều đủ **hai** ký tự.

### Lỗi 2: Nghĩ rằng chỉ palindrome mới `true`

In [23]:
def sai_chi_palindrome(s):
    return len(s) >= 2 and s == s[::-1]


print("sai:", sai_chi_palindrome("leetcode"), "| đúng:", Solution().isSubstringPresent("leetcode"))

sai: False | đúng: True


**Vì sao sai:** chỉ cần **một cặp** chung (ở `"leetcode"` là `"ee"`), không cần cả chuỗi bằng chuỗi đảo. **Cách sửa:** kiểm tra theo từng cặp.

### Lỗi 3: Bỏ sót trường hợp hai ký tự liền nhau giống nhau

In [24]:
def sai_bo_sot_cap_trung(s):
    pairs = {s[i:i + 2] for i in range(len(s) - 1)}
    return any(p[::-1] in pairs and p[::-1] != p for p in pairs)      # SAI: loại bỏ "xx"


print("sai:", sai_bo_sot_cap_trung("aa"), "| đúng:", Solution().isSubstringPresent("aa"))
print("sai:", sai_bo_sot_cap_trung("leetcode"), "| đúng:", Solution().isSubstringPresent("leetcode"))

sai: False | đúng: True
sai: False | đúng: True


**Vì sao sai:** người viết nghĩ "cặp đảo phải là một cặp **khác**", nhưng `"ee"` đảo vẫn là `"ee"` và **vẫn là chuỗi con chung** của `s` và `reverse(s)`. **Cách sửa:** không loại trường hợp `p[::-1] == p`.

### Lỗi 4: Quên chuỗi một ký tự

In [25]:
def sai_chuoi_mot_ky_tu(s):
    return s[0] == s[-1] or s[:2] in s[::-1]      # code "đại khái": với "a" cho True


print("sai:", sai_chuoi_mot_ky_tu("a"), "| đúng:", Solution().isSubstringPresent("a"))

sai: True | đúng: False


**Vì sao sai:** chuỗi một ký tự **không có chuỗi con độ dài 2**, đáp án phải `False`. Code kiểm tra `s[0] == s[-1]` coi `"a"` là hợp lệ. **Cách sửa:** chỉ xét các cặp liền nhau thật sự (`range(len(s) - 1)` rỗng khi `len(s) = 1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "chuỗi con của `s` và của `reverse(s)`": nhớ **đảo thứ tự cặp**: `xy` trong `reverse(s)` ⇔ `yx` trong `s`. Biến bài về **một** chuỗi.
- Chuỗi con **độ dài cố định nhỏ** (`2`, `3`) ⇒ dùng `set` các chuỗi con thay vì kiểm tra `in` lặp.
- Bảng chữ cái nhỏ (`26`) ⇒ bảng `26 x 26` hoặc bitmask thay cho băm chuỗi.
- Với cắt lát `s[i:i+k]`, nhớ **dừng ở `n - k`** (`range(n - k + 1)`), nếu không sẽ có lát cắt ngắn.
- Đừng nhầm "có cặp chung" với "là palindrome".

## 15. Biến thể của bài

1. **Chuỗi con độ dài `k` bất kỳ** (không chỉ `2`): `s[i:i + k]` và kiểm tra đảo của nó.
2. **Trả về cặp tìm được** (hoặc tất cả các cặp chung).
3. **Longest Palindromic Substring (5)**: tìm chuỗi con đối xứng dài nhất.
4. **Count Substrings That Are Palindromes (647)**.
5. **Chuỗi con chung dài nhất của `s` và `reverse(s)`** (liên quan tới palindrome).

In [26]:
def has_common_k(s, k):
    r = s[::-1]
    subs_s = {s[i:i + k] for i in range(len(s) - k + 1)}
    subs_r = {r[i:i + k] for i in range(len(r) - k + 1)}
    return bool(subs_s & subs_r)


def common_pairs(s):
    r = s[::-1]
    return sorted({s[i:i + 2] for i in range(len(s) - 1)} & {r[i:i + 2] for i in range(len(r) - 1)})


print(has_common_k("leetcode", 2), has_common_k("leetcode", 3))      # True False
print(common_pairs("abcba"))                                         # ['ab', 'ba', 'bc', 'cb']
assert has_common_k("leetcode", 2) is True
assert has_common_k("leetcode", 3) is False
assert common_pairs("abcba") == ["ab", "ba", "bc", "cb"]
assert has_common_k("abcd", 2) is False

True False
['ab', 'ba', 'bc', 'cb']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `pair in reverse` | đảo chuỗi, kiểm tra từng cặp | `O(n²)` | `O(n)` | rõ nhất, hợp `n` nhỏ |
| 2. So ký tự bằng tay | hai vòng lồng | `O(n²)` | `O(n)` | học cơ chế bên trong `in` |
| 3. Hai tập rồi giao | `pairs(s) & pairs(reverse)` | `O(n)` | `O(n)` | bám sát đề, nhanh |
| 4. Một tập: `xy` cần `yx` | không cần chuỗi đảo | `O(n)` | `O(n)` | **nên nộp** |
| 5. Bảng `26 x 26` | đánh dấu cặp `(a, b)` | `O(n)` | `O(1)` | cần bộ nhớ cố định, bảng chữ cái nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"hello"[::-1]`, `"abc"[1:3]`, `"ab" in "cabd"`, `Solution().isSubstringPresent("xyz")`, `Solution().isSubstringPresent("xyyz")`.

**Bài 2 (tính tay).** Với `s = "abcdcba"` liệt kê các cặp của `s` và của `reverse(s)`, tìm các cặp chung và đáp án. Làm tương tự cho `s = "acbd"`.

**Bài 3 (điền TODO).** Hoàn thành `present_todo(s)` theo Cách 4 (một tập các cặp).

In [27]:
def present_todo(s):
    # TODO 1: tạo tập pairs gồm mọi chuỗi con độ dài 2 của s (i từ 0 tới len(s) - 2)
    # TODO 2: nếu có cặp p mà p[::-1] cũng thuộc pairs thì trả True
    # TODO 3: ngược lại trả False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `present_table2(s)` theo Cách 5 (bảng `26 x 26`), **không** dùng `set` hay chuỗi đảo.

**Bài 5 (biến thể / mini challenge).** (a) Viết `present_k(s, k)` kiểm tra có **chuỗi con độ dài `k`** chung của `s` và `reverse(s)` hay không (dùng `set`). (b) Viết `longest_common_with_reverse(s)` trả về **độ dài lớn nhất `k`** mà `present_k(s, k)` là `True` (nếu không có `k >= 1` nào thì `0`); đối chiếu với brute force.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"olleh"`, `"bc"`, `True`, `False`, `True` (có `yy`). Bài 3: `any(p[::-1] in pairs for p in pairs)`.
- **Mức 2 (vừa):** Bài 2: `s = "abcdcba"` là palindrome ⇒ tập cặp của `reverse(s)` trùng tập cặp của `s` ⇒ `true`. `s = "acbd"`: cặp `ac, cb, bd`; reverse `dbca`: `db, bc, ca`; không chung ⇒ `false`. Bài 4: `seen[a][b] = True` rồi kiểm tra `seen[b][a]`.
- **Mức 3 (sâu):** Bài 5(a): `{s[i:i+k] for i in range(len(s) - k + 1)}` cho cả `s` và `r`, rồi giao. 5(b): thử `k` từ `len(s)` giảm dần; nhớ rằng `k = 1` luôn `True` (mọi ký tự của `s` có trong `reverse(s)`), nên đáp án luôn `>= 1`, và `len(s)` nếu `s` là palindrome.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "hello"[::-1] == "olleh"
assert "abc"[1:3] == "bc"
assert ("ab" in "cabd") is True
assert Solution().isSubstringPresent("xyz") is False
assert Solution().isSubstringPresent("xyyz") is True

# Bài 2
assert Solution().isSubstringPresent("abcdcba") is True
assert Solution().isSubstringPresent("acbd") is False


# Bài 3
def present_todo_ans(s):
    pairs = {s[i:i + 2] for i in range(len(s) - 1)}
    for p in pairs:
        if p[::-1] in pairs:
            return True
    return False


for s, expected in examples:
    assert present_todo_ans(s) == expected


# Bài 4
def present_table2(s):
    seen = [[False] * 26 for _ in range(26)]
    for i in range(len(s) - 1):
        a, b = ord(s[i]) - 97, ord(s[i + 1]) - 97
        seen[a][b] = True
        if seen[b][a]:
            return True
    return False


for s, expected in examples:
    assert present_table2(s) == expected


# Bài 5(a)
def present_k(s, k):
    r = s[::-1]
    return bool({s[i:i + k] for i in range(len(s) - k + 1)} & {r[i:i + k] for i in range(len(r) - k + 1)})


assert present_k("leetcode", 2) is True
assert present_k("leetcode", 3) is False
assert present_k("abcd", 1) is True


# Bài 5(b)
def longest_common_with_reverse(s):
    for k in range(len(s), 0, -1):
        if present_k(s, k):
            return k
    return 0


def longest_common_bruteforce(s):
    r = s[::-1]
    best = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            if s[i:j] in r:
                best = max(best, j - i)
    return best


assert longest_common_with_reverse("abcba") == 5
assert longest_common_with_reverse("abcd") == 1
rng = random.Random(9)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    assert longest_common_with_reverse(s) == longest_common_bruteforce(s)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chuỗi con độ dài 2 chung với chuỗi đảo ⇒ tập cặp, tìm cặp đảo)

```python
class Solution:
    def isSubstringPresent(self, s: str) -> bool:
        pairs = {s[i:i + 2] for i in range(len(s) - 1)}
        return any(p[::-1] in pairs for p in pairs)
```

**Checklist**

- [ ] Vòng lặp tạo cặp dừng ở `len(s) - 1` (không có lát cắt ngắn hơn 2)?
- [ ] Đã hiểu `xy` trong `reverse(s)` ⇔ `yx` trong `s`?
- [ ] Hai ký tự liền nhau giống nhau (`"ee"`) được tính là `true`?
- [ ] Chuỗi một ký tự trả `false`?
- [ ] Không nhầm với "s là palindrome"?
- [ ] Skeleton đúng `class Solution` / `def isSubstringPresent(self, s: str) -> bool`?